# Data Analysis Process — Coding Challenges

**Topic:** 09 Data Analysis Process · **Level:** 🟠 Intermediate · **Type:** CHALLENGES

Twenty minutes decides a lot in a data role. This notebook is a **ten-problem drill** — ten small coding challenges, each one a *screening* question recycled from real interview pipelines: define a framing brief, build a null-rate report, standardise messy columns, coerce dodgy dates, deduplicate on a normalised key, flag outliers with an IQR fence, melt untidy tables, profile with groupby, test missingness dependence, and validate a dataset. Every problem ends with an `assert`, so the notebook **grades itself**: green means correct, red means re-read the challenge.

## How to use this notebook

Work each challenge in order and **write the solution before you look at the next cell** — these are meant to be attempted cold, the way a screen does it. Read the one-line brief, recall the shape of the solution (which function, which parameter, which one-line idiom), type it, run it, and let the assert be the judge. If a challenge takes longer than ten minutes, that is your study list: re-do it, and re-do it again tomorrow.

## The engine of a good answer

Every challenge is designed to reward a single idiomatic move — `isna().mean()` for null-rates, a list comprehension for name cleaning, `errors='coerce'` for dates, `drop_duplicates` on a normalised key, `quantile` for the IQR, `melt` for untidiness, `agg` for profiling, `groupby` + `apply` for dependence, and a named validation function for the report. Notice what the honest answers share: small, typed, readable code that states its assumption and checks its own output. That is the fingerprint these challenges screen for.


## Setup

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
np.random.seed(0)
print('pandas', pd.__version__)

## Challenge 1

Write a framing brief and assert no field is empty.

## Challenge 1 — the framing brief

The first challenge is nearly philosophical: a **framing brief** is a dict of exactly five fields — `question`, `metric`, `population`, `window`, and `decision` — the irreducible answer to *what are we actually deciding here?*. Before any data science, analysts write the brief so the analysis has a spine.

The assert is the whole point: `assert all(brief.values())` demands that *no field is empty*. Two lessons hide in one line. First, `all()` on the dict's `.values()` is the idiomatic "every field populated" check — a one-line data-quality audit. Second, an analyst who starts with the question (what do we decide, on what metric, over what population and window) is an analyst who will not build the wrong model. If the brief has a hole, the analysis inherits it.

The challenge is solved in four keystrokes, but it screens for judgment: know your question before you touch your data.


In [ ]:
brief = {'question': 'q', 'metric': 'm', 'population': 'p',
         'window': 'w', 'decision': 'd'}
assert all(brief.values())
print(len(brief), 'fields complete')

## Challenge 2

Build a null-rate report for a DataFrame.

## Challenge 2 — the null-rate report

A **null-rate report** answers the first question asked of any real table: *where is the data missing?* The one-liner `df.isna().mean().round(3)` collapses the entire frame into per-column missing *fractions* — `a` at 1/3, `b` untouched at zero.

Two idioms carry it. `isna()` builds the boolean mask, `mean()` turns each column's `True`s into a proportion (count of missing over row count — not `sum()/len`, which would be the same thing spelled the hard way), and `round(3)` keeps the report readable. The assert then verifies the semantics: column `a` must have a nonzero rate, column `b` must be clean.

Why this challenge matters in screens: everyone can count `NaN`s, but the *rate* — standardized across columns — is what tells you *how* bad. A column that is 80% missing asks different questions than one missing 2%, and `isna().mean()` computes that on every column in one pass.


In [ ]:
import pandas as pd
df = pd.DataFrame({'a': [1, None, 3], 'b': [1, 2, 3]})
rep = df.isna().mean().round(3)
assert rep['a'] > 0 and rep['b'] == 0
print(rep.to_dict())

## Challenge 3

Standardise messy column names.

## Challenge 3 — standardising messy names

Real datasets ship with sloppy headers — `Order ID` with a space, `' Amount '` padded with blanks — and the third challenge is the canonical fix: a single list comprehension that runs three transforms over the columns in order: `strip()` removes the padding, `.lower()` standardises case, and `replace(' ', '_')` turns the gap into an underscore.

The whole clean happens in one line: `df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]`. Then the assert pins the exact outcome: `['order_id', 'amount']`.

The interview reflex being screened is *where* the cleaning belongs: column names are cleaned at the **source** (the columns object), not downstream. A DataFrame whose headers are consistently lowercase, trimmed, and underscored is one that never makes you guess `df['Order ID ']` again — and one comprehension beats ten manual renames. The same chain scales: if headers also carry stray periods or newlines, the comprehension simply grows another step, because cleaning column names is always a small pipeline, never a one-off hack.


In [ ]:
import pandas as pd
df = pd.DataFrame({'Order ID': [1], ' Amount ': [2]})
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
assert list(df.columns) == ['order_id', 'amount']
print(list(df.columns))

## Challenge 4

Coerce a mixed date column and count the failures.

## Challenge 4 — coercing a mixed date column

Challenge 4 is about what pandas *should* do when a column of dates contains a non-date: `pd.to_datetime(pd.Series(['2026-01-05', 'not a date']), errors='coerce')` converts what it can and turns the failure into `NaN`.

`errors='coerce'` is the whole lesson — the parameter that separates reading a date column from *surviving* a date column. Without it, one bad string raises and the pipeline dies mid-read; with it, the good values parse and the bad ones become explicit missing markers you can count.

Counting is exactly what the assert demands: `s.isna().sum() == 1` proves exactly one bad date. The screen is testing whether you know the *parameter*, not the function — interviews lean on `errors='coerce'` with dates and `astype` precisely because a candidate who reaches for it unprompted has been bitten by real data before. The companion habit is to keep the result visible: a quick `isna().sum()` after parsing tells you exactly how many rows you traded for a readable column.


In [ ]:
import pandas as pd
s = pd.to_datetime(pd.Series(['2026-01-05', 'not a date']), errors='coerce')
assert s.isna().sum() == 1
print('bad dates:', int(s.isna().sum()))

## Challenge 5

Deduplicate on a normalised key.

## Challenge 5 — deduplicating on a normalised key

Duplicate rows are rarely byte-identical — the same customer wrote `A@x.com` and `a@X.com`. So Challenge 5 rewrites the rule: deduplicate on a **normalised key**. `df['key'] = df['email'].str.lower()` collapses emails to a comparison-safe form, then `df.drop_duplicates('key')` keeps one row per real identity, reducing three rows to two.

The assert checks the outcome, but the lesson is the *normalisation* step. `drop_duplicates` alone is the weak answer; what makes the challenge respectable is realising that case (`A@x.com` vs `a@X.com`) is a data-quality problem that must be folded into the identity itself before deduplication means anything.

This pattern recurs everywhere in production — keys normalised by lowercasing, trimming, stripping zeros or timezones before any join or de-dup. The screen is not testing `drop_duplicates`; it is testing whether you know that *deduplication is only as honest as its key*. The same instinct: decide what really identifies a row, fold any variation into it before comparing, then de-dup. That is what turns a brittle check into a dependable one.


In [ ]:
import pandas as pd
df = pd.DataFrame({'email': ['A@x.com', 'a@X.com', 'b@x.com']})
df['key'] = df['email'].str.lower()
clean = df.drop_duplicates('key')
assert len(clean) == 2
print(clean)

## Challenge 6

Flag values outside the IQR fence.

## Challenge 6 — the IQR outlier fence

Outliers get flagged before they get removed. Challenge 6 fences a numeric series with the **IQR rule**: `q1, q3 = s.quantile([0.25, 0.75])` computes the quartiles, `f = q3 + 1.5 * (q3 - q1)` builds the upper fence, and `(s > f).sum()` counts the values beyond it. With `[1, 2, 3, 4, 100]`, exactly one point — the 100 — flags.

Three idioms matter. `quantile([0.25, 0.75])` unpacks two quartiles in one call. The `1.5 *` multiplier is the standard convention for "extreme" in Tukey's rule — not magic, a widely-used threshold. And the mask `s > f` is a *boolean filter* that both counts (`sum()`) and can later select (`s[s > f]`).

The screen wants you to know the fence formula cold, and to frame outliers as *flag first, judge later*. Outlier removal is a decision; flagging is just measurement, and the assert keeps you to measurement.


In [ ]:
import pandas as pd
s = pd.Series([1, 2, 3, 4, 100])
q1, q3 = s.quantile([0.25, 0.75])
f = q3 + 1.5 * (q3 - q1)
assert (s > f).sum() == 1
print('flagged:', s[s > f].tolist())

## Challenge 7

Melt an untidy table and verify the row count.

## Challenge 7 — melting an untidy table

Untidy data hides values *in column names*: quarters `q1`, `q2` sitting side by side. Challenge 7 fixes it with a single idiom — `w.melt(id_vars='city')` — which pivots the quarter-columns **down** into rows: two cities × two quarters becomes **four rows** of `city`, `variable` (the quarter), and `value`.

The assert `len(t) == 4` is the arithmetic of tidiness: rows = id_values × measured columns. That is the hallmark of *tidy data* — each column is a variable, each row an observation — and `melt` is the interview shorthand for producing it.

What the screen actually measures: recognition that a table whose headers contain data must be **reshaped, not re-typed**. Candidates who reach for `melt` on sight are candidates who have met wide formats in the wild — the classic `pd.wide_to_long` family lives in the same muscle memory. Once melted, the quarter and value drop into ordinary columns, so filtering, grouping, or plotting need no special casing: the tidy layout is the point.


In [ ]:
import pandas as pd
w = pd.DataFrame({'city': ['a', 'b'], 'q1': [1, 2], 'q2': [3, 4]})
t = w.melt(id_vars='city')
assert len(t) == 4
print(t)

## Challenge 8

Compute a groupby profile with three aggregates.

## Challenge 8 — the groupby profile

Profiling means *summarising per group*, and Challenge 8 builds the report in one brace: `df.groupby('g')['v'].agg(['count', 'mean', 'max'])` splits by `g`, pulls column `v`, and applies three aggregates at once — the exact form a null or distribution profile takes in meaty code.

The chained accessors are the lesson: `groupby(...)` creates the groups, `['v']` narrows to the measured column, `.agg([...])` fans out into a multi-column result with each statistic as a column. The assert confirms the arithmetic — group `a` contains two rows — but the real signal is whether the candidate can collapse a frame into a *comparison* without a manual loop.

This is the single most reused idiom in reporting: totals per category, averages per segment, counts per cohort — all `groupby` + `agg`. Screens like this one because `child` code you will write every day of a data job is the strongest predictor of the job itself.


In [ ]:
import pandas as pd
df = pd.DataFrame({'g': ['a', 'a', 'b'], 'v': [1, 3, 5]})
out = df.groupby('g')['v'].agg(['count', 'mean', 'max'])
assert out.loc['a', 'count'] == 2
print(out)

## Challenge 9

Detect whether missingness depends on another column.

## Challenge 9 — dependence of missingness

Challenge 9 asks a genuinely diagnostic question: *does missingness depend on another column?* The template: `df.assign(old=df['age'] > 60).groupby('old')['inc'].apply(lambda s: s.isna().mean())` — it flags rows as old/young, groups, and computes the missing *rate* of `inc` within each band. If the rates differ meaningfully, missingness is **not random** — it correlates with age — and that fact changes how you interpret the data.

The mechanics: `assign` adds the derived grouping column; `groupby('old')` splits; `.apply(lambda s: s.isna().mean())` computes the semantics per slice. The assert pins the extreme case — every `old` row is missing `inc`, so its rate is exactly 1.0.

This is the *MCAR vs MNAR* question in miniature: missing-completely-at-random versus missing-not-at-random. Screens include it to see whether a candidate can *test* missingness rather than blindly `dropna()` — missing data that depends on other columns is biased data, and the flag is the warning. The practical follow-up is deciding how to act on it, but the interview only wants to see the test itself.


In [ ]:
import numpy as np, pandas as pd
df = pd.DataFrame({'age': [20, 70, 72], 'inc': [1, np.nan, np.nan]})
r = df.assign(old=df['age'] > 60).groupby('old')['inc'].apply(
    lambda s: s.isna().mean())
assert r[True] == 1.0
print(r.to_dict())

## Challenge 10

Write a validation function that returns a pass or fail report.

## Challenge 10 — the validation report

The final challenge is the professional's move: bundle data-quality checks into a **named function** that returns a pass/fail report instead of scattering asserts through the code. `validate(df)` returns a dict — `{'rows': len(df) > 0, 'unique_id': df['id'].is_unique}` — and `assert all(rep.values())` demands both checks.

Two lessons carry it. **Functions, not free code** — a `validate` function is importable, reusable, and testable in isolation; real pipelines wrap exactly this pattern so every input is vetted at the boundary. **Reports, not asserts** — returning named booleans (`unique_id`, `rows`) turns the check into *communication*: other analysts can read the report's keys and see precisely which gate failed.

And the screen's closing point: `.is_unique` is the idiomatic one-liner for duplicate-free keys, the same instinct that answered Challenge 5. The ten challenges have been training one muscle all along — honest checks, typed small. Run each function against a tiny sample before the real frame, and the report pattern guarantees no gate is ever silently skipped.


In [ ]:
import pandas as pd
def validate(df):
    return {'rows': len(df) > 0, 'unique_id': df['id'].is_unique}

df = pd.DataFrame({'id': [1, 2]})
rep = validate(df)
assert all(rep.values())
print(rep)

## After the session

You have now taken the ten-challenge screen — treat it like the timing does. **Time yourself**: ten minutes per challenge is a realistic screening pace, and the notebook's asserts give you an instant pass-fail on each one. Re-do the ones that took longer, and re-do them *again* tomorrow, because the second pass is where recall turns into reflex.

**Sharpen your study with evidence.** For each slow problem, write one line naming the gap: did you forget the parameter (`errors='coerce'`), mispick the tool (`melt` instead of a manual loop), or miss the reduction (`isna().mean()` instead of `sum()/len()`)? A named gap is a fixable gap; vague discomfort is not.

**Grow the drill.** The strongest way to use these ten is to re-engineer them: change the DataFrame, change the column names, add a twist (an extra null, a malicious string, a second group), and solve again. Screening problems reappear everywhere — on whiteboards, in take-homes, in your own future notebooks — and a well-drilled armature of these idioms will answer them all.


## Summary

Ten challenges, ten idiomatic moves: the framing brief, `isna().mean()` null-rates, name standardisation by comprehension, `errors='coerce'` dates, normalise-then-`drop_duplicates`, the IQR fence, `melt`, `groupby` + `agg`, missingness-by-group tests, and a named `validate` report. Every one ended in an `assert` — so the notebook graded itself and left nothing to guess.

The through-line is the same: **state the check, run it, let the data answer**. Each idiom is small because each is one honest step — and ten honest steps, re-drilled under a clock, are exactly what a ten-minute screen looks like. The session also leaves you a reusable toolkit: each challenge is one small carve-out you can paste into a real notebook tomorrow.
